# SampleManager Tutorial: Node-Based Filtering Across Heterogeneous Datasets

This tutorial introduces `SampleManager`, a flexible node-based system for filtering samples across multiple datasets with varying experimental condition structures.

## Why SampleManager?

Traditional table-based approaches struggle with heterogeneous metadata:
- Different datasets structure conditions differently (e.g., `media.carbon_source` vs `environmental_conditions.media.carbon_source`)
- Missing fields and optional properties vary by dataset
- Multi-level hierarchies (repo/config/field) require complex joins

**SampleManager** solves this by:
- Representing each sample as a node with dynamically discovered properties
- Supporting flexible MongoDB-style queries across heterogeneous structures
- Enabling cross-dataset filtering and set operations
- Respecting hierarchical overrides (field > config > repo)

## Key Concepts

- **SampleNode**: A sample with flattened properties (experimental conditions + metadata)
- **ActiveSet**: A filtered collection of sample IDs supporting set operations
- **Query Language**: MongoDB-style filters with operators like `$contains`, `$gte`, `$or`
- **Property Discovery**: No hardcoded schemas - properties are discovered from the data

## 1. Setup and Basic Usage

In [35]:
from tfbpapi.datainfo.sample_manager import SampleManager, SampleNode
from tfbpapi.datainfo import DataCard
import json

# Initialize manager
manager = SampleManager()
print(manager)

SampleManager(0 datasets, 0 samples)


## 2. Understanding Property Discovery

Let's explore how SampleManager discovers properties from a DataCard without hardcoded schemas.

In [36]:
# Load a DataCard to see its experimental conditions structure
card = DataCard('BrentLab/harbison_2004')

# Get repo-level conditions
repo_conditions = card.get_experimental_conditions('harbison_2004')
print("Repo-level conditions:")
print(json.dumps(repo_conditions, indent=2, default=str))

Repo-level conditions:
{}


In [37]:
# Get field-level condition definitions
field_defs = card.get_field_definitions('harbison_2004', 'condition')

# Show one condition definition
print("\nField-level definition for 'YPD':")
print(json.dumps(field_defs.get('YPD', {}), indent=2, default=str))


Field-level definition for 'YPD':
{
  "description": "Rich media baseline condition",
  "temperature_celsius": 30,
  "cultivation_method": "unspecified",
  "growth_phase_at_harvest": {
    "od600": 0.8
  },
  "media": {
    "name": "YPD",
    "carbon_source": [
      {
        "compound": "D-glucose",
        "concentration_percent": 2
      }
    ],
    "nitrogen_source": [
      {
        "compound": "yeast_extract",
        "concentration_percent": 1
      },
      {
        "compound": "peptone",
        "concentration_percent": 2
      }
    ]
  }
}


### How Properties Are Flattened

SampleManager recursively flattens nested structures using dot notation:

```python
# Original nested structure:
{
  "environmental_conditions": {
    "media": {
      "name": "YPD",
      "carbon_source": [{"compound": "D-glucose", "concentration_percent": 2}]
    },
    "temperature_celsius": 30
  }
}

# Becomes flattened properties:
{
  "environmental_conditions.media.name": "YPD",
  "environmental_conditions.media.carbon_source": "D-glucose",  # Simple string
  "_environmental_conditions.media.carbon_source_structured": [{...}],  # Full structure
  "environmental_conditions.temperature_celsius": 30
}
```

**Key Points:**
- Nested dicts become dot-notation keys
- Lists of dicts (compounds) get both simple and `_structured` versions
- No hardcoded field names - discovers whatever exists
- Different datasets can have different structures

## 3. Creating Sample Nodes Manually

Before we load from DataCard, let's manually create nodes to understand the structure.

In [38]:
from tfbpapi.datainfo.sample_manager import SampleNode, ConditionFlattener

# Simulate experimental conditions from different levels
repo_conditions = {
    "environmental_conditions": {
        "temperature_celsius": 30,
        "cultivation_method": "liquid_culture"
    },
    "strain_background": "BY4741"
}

field_conditions = {
    "environmental_conditions": {
        "media": {
            "name": "YPD",
            "carbon_source": [
                {"compound": "D-glucose", "concentration_percent": 2}
            ],
            "nitrogen_source": [
                {"compound": "yeast_extract", "concentration_percent": 1},
                {"compound": "peptone", "concentration_percent": 2}
            ]
        }
    }
}

# Flatten conditions
properties, sources = ConditionFlattener.flatten_conditions(
    repo_conditions, None, field_conditions
)

print("Flattened properties:")
for key, value in sorted(properties.items()):
    source = sources.get(key, 'unknown')
    # Skip structured versions for cleaner output
    if not key.startswith('_'):
        print(f"  {key}: {value} (from {source})")

Flattened properties:
  environmental_conditions.cultivation_method: liquid_culture (from repo)
  environmental_conditions.media.carbon_source: D-glucose (from field)
  environmental_conditions.media.name: YPD (from field)
  environmental_conditions.media.nitrogen_source: yeast_extract, peptone (from field)
  environmental_conditions.temperature_celsius: 30 (from repo)
  strain_background: BY4741 (from repo)


In [39]:
# Create a sample node
node = SampleNode(
    sample_id="sample_001",
    repo_id="BrentLab/harbison_2004",
    config_name="harbison_2004",
    properties=properties,
    property_sources=sources
)

print(f"Node: {node}")
print(f"Global ID: {node.global_id()}")
print(f"\nSample properties:")
print(f"  Temperature: {node.get_property('environmental_conditions.temperature_celsius')}")
print(f"  Carbon source: {node.get_property('environmental_conditions.media.carbon_source')}")

Node: SampleNode(BrentLab/harbison_2004:harbison_2004:sample_001, 8 properties)
Global ID: BrentLab/harbison_2004:harbison_2004:sample_001

Sample properties:
  Temperature: 30
  Carbon source: D-glucose


## 4. Filtering with the Query Language

SampleManager uses MongoDB-style queries for flexible filtering.

In [40]:
from tfbpapi.datainfo.sample_manager import SampleFilter

# Simple equality
query1 = {"environmental_conditions.temperature_celsius": 30}
print(f"Query 1 matches: {SampleFilter.matches(node, query1)}")

# Contains check (case-insensitive)
query2 = {"environmental_conditions.media.carbon_source": {"$contains": "glucose"}}
print(f"Query 2 matches: {SampleFilter.matches(node, query2)}")

# Numeric comparison
query3 = {"environmental_conditions.temperature_celsius": {"$gte": 25, "$lte": 35}}
print(f"Query 3 matches: {SampleFilter.matches(node, query3)}")

# Logical OR
query4 = {
    "$or": [
        {"environmental_conditions.media.carbon_source": {"$contains": "galactose"}},
        {"environmental_conditions.media.carbon_source": {"$contains": "glucose"}}
    ]
}
print(f"Query 4 matches: {SampleFilter.matches(node, query4)}")

Query 1 matches: True
Query 2 matches: True
Query 3 matches: True
Query 4 matches: True


### Available Query Operators

| Operator | Description | Example |
|----------|-------------|----------|
| `$eq` | Equal (default) | `{"temp": 30}` or `{"temp": {"$eq": 30}}` |
| `$ne` | Not equal | `{"temp": {"$ne": 30}}` |
| `$gt` | Greater than | `{"temp": {"$gt": 30}}` |
| `$gte` | Greater than or equal | `{"temp": {"$gte": 30}}` |
| `$lt` | Less than | `{"temp": {"$lt": 30}}` |
| `$lte` | Less than or equal | `{"temp": {"$lte": 30}}` |
| `$in` | In list | `{"strain": {"$in": ["BY4741", "W303"]}}` |
| `$nin` | Not in list | `{"strain": {"$nin": ["BY4741"]}}` |
| `$contains` | String contains (case-insensitive) | `{"carbon_source": {"$contains": "glucose"}}` |
| `$exists` | Field exists | `{"temperature": {"$exists": true}}` |
| `$and` | Logical AND | `{"$and": [{...}, {...}]}` |
| `$or` | Logical OR | `{"$or": [{...}, {...}]}` |

## 5. Working with ActiveSets

ActiveSets represent filtered collections of samples and support set operations.

In [41]:
from tfbpapi.datainfo.sample_manager import ActiveSet

# Create some example active sets
set1 = ActiveSet(
    sample_ids={"BrentLab:harbison_2004:sample_001", "BrentLab:harbison_2004:sample_002", "BrentLab:harbison_2004:sample_003"},
    name="glucose_samples",
    description="Samples grown on glucose"
)

set2 = ActiveSet(
    sample_ids={"BrentLab:harbison_2004:sample_002", "BrentLab:harbison_2004:sample_003", "BrentLab:harbison_2004:sample_004"},
    name="heat_stress_samples",
    description="Samples with heat stress"
)

print(f"Set 1: {set1}")
print(f"Set 2: {set2}")

Set 1: ActiveSet(name=glucose_samples, size=3)
Set 2: ActiveSet(name=heat_stress_samples, size=3)


In [42]:
# Set operations

# Union - samples in either set
union = set1.union(set2, name="glucose_or_heat")
print(f"Union: {union}")
print(f"  Sample IDs: {union.to_sample_ids()}")

# Intersection - samples in both sets
intersection = set1.intersection(set2, name="glucose_and_heat")
print(f"\nIntersection: {intersection}")
print(f"  Sample IDs: {intersection.to_sample_ids()}")

# Difference - samples in set1 but not set2
difference = set1.difference(set2, name="glucose_no_heat")
print(f"\nDifference: {difference}")
print(f"  Sample IDs: {difference.to_sample_ids()}")

Union: ActiveSet(name=glucose_or_heat, size=4)
  Sample IDs: ['BrentLab:harbison_2004:sample_001', 'BrentLab:harbison_2004:sample_002', 'BrentLab:harbison_2004:sample_003', 'BrentLab:harbison_2004:sample_004']

Intersection: ActiveSet(name=glucose_and_heat, size=2)
  Sample IDs: ['BrentLab:harbison_2004:sample_002', 'BrentLab:harbison_2004:sample_003']

Difference: ActiveSet(name=glucose_no_heat, size=1)
  Sample IDs: ['BrentLab:harbison_2004:sample_001']


## 6. Real-World Example: Exploring Harbison 2004

Now let's work with real data. Note that we're manually creating nodes here since the `load_from_datacard` method needs to be implemented. This demonstrates the intended workflow.

In [43]:
# For this tutorial, we'll manually create nodes from DataCard information
# In production, you would use: manager.load_from_datacard("BrentLab/harbison_2004", "harbison_2004")

from tfbpapi.datainfo import DataCard

card = DataCard('BrentLab/harbison_2004')

# Get repo-level and field-level conditions
repo_conds = card.get_experimental_conditions('harbison_2004')
field_defs = card.get_field_definitions('harbison_2004', 'condition')

print(f"Found {len(field_defs)} condition definitions")
print(f"Conditions: {sorted(field_defs.keys())}")

Found 14 condition definitions
Conditions: ['Acid', 'Alpha', 'BUT14', 'BUT90', 'GAL', 'H2O2Hi', 'H2O2Lo', 'HEAT', 'Pi-', 'RAFF', 'RAPA', 'SM', 'Thi-', 'YPD']


In [44]:
# Manually create nodes for each condition (simulating samples)
# This demonstrates what load_from_datacard will do automatically

demo_manager = SampleManager()

for condition_name, condition_def in field_defs.items():
    # Flatten conditions
    properties, sources = ConditionFlattener.flatten_conditions(
        repo_conds, None, condition_def
    )
    
    # Create node (using condition name as sample_id for demo)
    node = SampleNode(
        sample_id=condition_name,
        repo_id="BrentLab/harbison_2004",
        config_name="harbison_2004",
        properties=properties,
        property_sources=sources
    )
    
    # Add to manager's collection
    demo_manager._collection.add_node(node)

print(f"\nLoaded {demo_manager._collection.count_total_nodes()} sample nodes")
print(demo_manager)


Loaded 14 sample nodes
SampleManager(1 datasets, 14 samples)


In [45]:
# Explore what properties are available
sample_node = demo_manager.get_sample("BrentLab/harbison_2004:harbison_2004:YPD")

print("Properties available in YPD sample:")
for key in sorted(sample_node.properties.keys()):
    if not key.startswith('_'):  # Skip structured versions
        print(f"  {key}: {sample_node.properties[key]}")

Properties available in YPD sample:
  cultivation_method: unspecified
  description: Rich media baseline condition
  growth_phase_at_harvest.od600: 0.8
  media.carbon_source: D-glucose
  media.name: YPD
  media.nitrogen_source: yeast_extract, peptone
  temperature_celsius: 30


## 7. Cross-Condition Filtering

Now we can filter samples based on their experimental conditions.

In [46]:
# Find all samples with D-glucose as carbon source
glucose_samples = demo_manager.filter_all({
    "environmental_conditions.media.carbon_source": {"$contains": "D-glucose"}
}, name="glucose_conditions")

print(f"Found {len(glucose_samples)} conditions with D-glucose")
print(f"Conditions: {[sid.split(':')[-1] for sid in glucose_samples.to_sample_ids()]}")

Found 0 conditions with D-glucose
Conditions: []


In [47]:
# Find samples with alternative carbon sources
alt_carbon = demo_manager.filter_all({
    "$or": [
        {"environmental_conditions.media.carbon_source": {"$contains": "galactose"}},
        {"environmental_conditions.media.carbon_source": {"$contains": "raffinose"}}
    ]
}, name="alternative_carbon")

print(f"\nFound {len(alt_carbon)} conditions with alternative carbon sources")
print(f"Conditions: {[sid.split(':')[-1] for sid in alt_carbon.to_sample_ids()]}")


Found 0 conditions with alternative carbon sources
Conditions: []


In [48]:
# Find samples with media additives (like butanol)
with_additives = demo_manager.filter_all({
    "environmental_conditions.media.additives": {"$contains": "butanol"}
}, name="additive_conditions")

print(f"\nFound {len(with_additives)} conditions with additives")
print(f"Conditions: {[sid.split(':')[-1] for sid in with_additives.to_sample_ids()]}")


Found 0 conditions with additives
Conditions: []


## 8. Property Distribution Analysis

In [49]:
# Analyze carbon source distribution
carbon_dist = demo_manager.get_property_distribution(
    "environmental_conditions.media.carbon_source"
)

print("Carbon source distribution:")
for value, count in sorted(carbon_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {value}: {count} samples")

Carbon source distribution:
  missing: 14 samples


In [50]:
# Analyze media names
media_dist = demo_manager.get_property_distribution(
    "environmental_conditions.media.name"
)

print("\nMedia type distribution:")
for value, count in sorted(media_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {value}: {count} samples")


Media type distribution:
  missing: 14 samples


## 9. Summary Information

In [51]:
# Get summary of loaded data
summary = demo_manager.get_summary()
print("\nSummary of loaded datasets:")
print(summary)


Summary of loaded datasets:
                  repo_id    config_name  sample_count  \
0  BrentLab/harbison_2004  harbison_2004            14   

                                          properties  
0  [description, temperature_celsius, cultivation...  


## 10. Key Takeaways

### Advantages of Node-Based Approach

1. **Flexible Schema**: No hardcoded field names - discovers whatever exists in the data
2. **Heterogeneity Handling**: Different datasets can have different structures (e.g., `media.carbon_source` vs `environmental_conditions.media.carbon_source`)
3. **Cross-Dataset Queries**: Filter samples across multiple datasets with varying structures
4. **Set Operations**: Build complex filters incrementally using union/intersection/difference
5. **Property Discovery**: Automatically flattens nested structures using dot notation
6. **Dual Representation**: Compound lists get both simple strings and structured versions

### How to Query Different Property Paths

Since different datasets structure their conditions differently, you need to use the actual property path:

```python
# Dataset A (harbison_2004): media nested under environmental_conditions
manager.filter_all({"environmental_conditions.media.carbon_source": {"$contains": "glucose"}})

# Dataset B (hypothetical): media at top level  
manager.filter_all({"media.carbon_source": {"$contains": "glucose"}})

# To query across both, use $or:
manager.filter_all({
    "$or": [
        {"environmental_conditions.media.carbon_source": {"$contains": "glucose"}},
        {"media.carbon_source": {"$contains": "glucose"}}
    ]
})
```

### Future: External Property Mappings

For cross-dataset harmonization, you can provide external YAML mappings:

```yaml
# property_mappings.yaml
carbon_source:
  paths:
    - environmental_conditions.media.carbon_source
    - media.carbon_source
  value_aliases:
    D-glucose: [glucose, dextrose, D-dextrose]
```

This would enable canonical queries like `{"carbon_source": {"$contains": "glucose"}}` to work across all datasets.

## Next Steps

- Implement `load_from_datacard()` to automatically load samples from HuggingFace
- Implement `load_from_duckdb()` for integration with HfQueryAPI
- Implement `export_to_duckdb()` to convert ActiveSets back to SQL views
- Add external property mapping support via YAML configuration
- Build UI for interactive filtering and set operations